# NBA customer-risk classifier — PyTorch + ONNX

Trains a small PyTorch feed-forward network on the same SQLite `customers` table used by the XGBoost variant (`../xgboost/01_train_xgboost_onnx.ipynb`), exports the trained network to ONNX, and registers **both** the PyTorch and ONNX artifacts with the CAI Model Registry via MLflow.

- **Label:** binary `is_high_risk = (risk_tier == 'HIGH')` — 2 output classes.
- **Features (8, fixed order):** `age`, `annual_income`, `existing_debt`, plus one-hot `emp_EMPLOYED / SELF_EMPLOYED / STUDENT / UNEMPLOYED / RETIRED`. Standardized with `StandardScaler` at train time; the same normalization is required at inference time (scaler stats logged as MLflow params).
- **Registry names:**
  - `nba-risk-pytorch` — raw PyTorch model artifact.
  - `nba-risk-onnx-pytorch` — ONNX artifact for the CAI Inference Service. Point `CLF_MODEL_ID` at this one.

Pattern is adapted from https://github.com/zoramt/aiinference/blob/main/examples/pytorch/iris_mlflow_pytorch_onnx.py.

In [ ]:
#****************************************************************************
# (C) Cloudera, Inc. 2020-2023
#  All rights reserved.
#
#  Applicable Open Source License: GNU Affero General Public License v3.0
#
#  NOTE: Cloudera open source products are modular software products
#  made up of hundreds of individual components, each of which was
#  individually copyrighted.  Each Cloudera open source product is a
#  collective work under U.S. Copyright Law. Your license to use the
#  collective work is as provided in your written agreement with
#  Cloudera.  Used apart from the collective work, this file is
#  licensed for your use pursuant to the open source license
#  identified above.
#
#  Author(s): Paul de Fusco
#***************************************************************************/

In [ ]:
# onnxscript is lazy-imported by torch.onnx.export in PyTorch >= 2.5 and is
# not always present on the CAI base runtime.
#
# We also uninstall any user-local mlflow install (from a prior
# ``pip install -r requirements.txt`` that pinned a newer version) so that
# ``import mlflow`` falls back to the CAI-shipped client, which is the one
# that speaks the CAI tracking server's protocol.  Restart the kernel after
# this cell the first time you run the notebook.
!pip install --quiet onnxscript
!pip uninstall --quiet --yes mlflow 2>/dev/null || true

In [ ]:
import os
import sys
from datetime import date

import mlflow
import mlflow.onnx
import mlflow.pytorch
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, classification_report, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Notebook lives in pytorch_train/; the app code (db.py, pii_datagen.py) lives
# in ../app.  Put that on sys.path so we can import it without installing the
# project as a package.
sys.path.insert(0, os.path.abspath(os.path.join("..", "app")))
from db import DB_PATH, get_conn, init_schema, seed_offers  # noqa: E402
from pii_datagen import _EMP_STATUSES, seed_customers  # noqa: E402

# Reproducibility.
torch.manual_seed(42)
np.random.seed(42)

print(f"MLflow version: {mlflow.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"ONNX version:    {onnx.__version__}")

In [ ]:
# Idempotently seed the SQLite DB from pii_datagen so this notebook is
# self-contained: rerun it and you always get the same 10k customers.
with get_conn() as conn:
    init_schema(conn)
    seed_offers(conn)
    seed_customers(conn, n=10_000, seed=42)

print(f"SQLite DB at: {DB_PATH}")

In [ ]:
USERNAME = os.environ.get("PROJECT_OWNER", "local")
DATE = date.today()
EXPERIMENT_NAME = f"nba-customer-risk-pytorch-{USERNAME}"
REGISTERED_MODEL_NAME_PYTORCH = "nba-risk-pytorch"
REGISTERED_MODEL_NAME_ONNX = "nba-risk-onnx-pytorch"

mlflow.set_experiment(EXPERIMENT_NAME)

## Load features from the `customers` table

Column order **must** match what `nba_app.py` sends at inference time, so we build both the numeric block and the one-hot block from a fixed list of employment-status categories imported from `pii_datagen`.

In [ ]:
with get_conn() as conn:
    df = pd.read_sql_query(
        "SELECT age, annual_income, existing_debt, employment_status, risk_tier "
        "FROM customers",
        conn,
    )

# Binary label: 1 if HIGH risk, else 0.  Two-class output keeps the ONNX
# output shape at [-1, 2] so nba_app.py's response parsing
# (outputs[1]["data"][1] = P(class=1)) is unchanged.
y = (df["risk_tier"] == "HIGH").astype(np.int64)

# Numeric block.
X_numeric = df[["age", "annual_income", "existing_debt"]].astype(np.float32)

# One-hot employment_status.  Iterate the fixed category list from pii_datagen
# so column order is deterministic across train time and inference time even
# if some categories are absent from a given sample.
X_onehot = pd.DataFrame(
    {
        f"emp_{status}": (df["employment_status"] == status).astype(np.float32)
        for status in _EMP_STATUSES
    }
)

X = pd.concat([X_numeric, X_onehot], axis=1)
FEATURE_COLUMNS = list(X.columns)

print(f"features ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")
print(f"label distribution: {y.value_counts().to_dict()}")

In [ ]:
test_size = 0.3
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

# StandardScaler is fit on the training block only and reused for test.  We
# fold scaling into the ONNX graph too — see the export cell below — so the
# deployed endpoint expects RAW (unscaled) features, matching what
# nba_app.py reads out of state["customer_record"].
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df.values).astype(np.float32)
X_test_scaled = scaler.transform(X_test_df.values).astype(np.float32)

X_train_tensor = torch.from_numpy(X_train_scaled)
X_test_tensor = torch.from_numpy(X_test_scaled)
y_train_tensor = torch.from_numpy(y_train.values.astype(np.int64))
y_test_tensor = torch.from_numpy(y_test.values.astype(np.int64))

print(f"train={X_train_tensor.shape} test={X_test_tensor.shape}")

## Model

Small feed-forward net: `Linear → ReLU → Dropout → Linear → ReLU → Dropout → Linear`. Two output units (class 0 = not high-risk, class 1 = high-risk). Trained with `CrossEntropyLoss` (applies softmax internally); the exported ONNX graph returns raw logits — the client applies softmax + argmax at inference time, matching the CAI reference pattern.

In [ ]:
class RiskClassifier(nn.Module):
    """Feed-forward binary classifier over the 8 customer-profile features."""

    def __init__(self, input_size: int, hidden_size: int = 32, num_classes: int = 2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)  # raw logits — CrossEntropyLoss handles softmax


print("Model architecture defined.")

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs: int):
    model.train()
    train_losses, train_accs = [], []
    for epoch in range(num_epochs):
        epoch_loss, correct, total = 0.0, 0, 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

        avg_loss = epoch_loss / max(1, len(train_loader))
        acc = 100.0 * correct / max(1, total)
        train_losses.append(avg_loss)
        train_accs.append(acc)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"epoch {epoch + 1:>3}/{num_epochs}  loss={avg_loss:.4f}  acc={acc:.2f}%")
    return train_losses, train_accs


def evaluate_model(model, test_loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            _, predicted = torch.max(model(batch_X), 1)
            preds.extend(predicted.cpu().numpy())
            targets.extend(batch_y.cpu().numpy())
    return preds, targets


print("Training helpers defined.")

In [ ]:
# Hyperparameters — small model, so training is fast even on CPU.
hidden_size = 32
learning_rate = 1e-3
num_epochs = 30
batch_size = 64
input_size = X_train_tensor.shape[1]  # 8

with mlflow.start_run():
    mlflow.log_param("hidden_size", hidden_size)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("num_epochs", num_epochs)
    mlflow.log_param("batch_size", batch_size)
    mlflow.log_param("test_size", test_size)
    mlflow.log_param("n_features", input_size)
    mlflow.log_param("feature_columns", ",".join(FEATURE_COLUMNS))
    # Persist scaler stats so the deploy notebook / app can reconstruct the
    # same normalization at inference time.
    mlflow.log_param("scaler_mean", ",".join(f"{v:.6f}" for v in scaler.mean_))
    mlflow.log_param("scaler_scale", ",".join(f"{v:.6f}" for v in scaler.scale_))

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = RiskClassifier(input_size=input_size, hidden_size=hidden_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print("Training...")
    train_losses, train_accs = train_model(
        model, train_loader, criterion, optimizer, num_epochs
    )

    print("\nEvaluating on test split...")
    preds, targets = evaluate_model(model, test_loader)
    test_accuracy = accuracy_score(targets, preds)
    test_recall = recall_score(targets, preds)

    print(f"\nTest accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test recall:   {test_recall * 100:.2f}%")
    print("\n" + classification_report(targets, preds, target_names=["low/med", "high"]))

    mlflow.log_metric("final_train_accuracy", train_accs[-1])
    mlflow.log_metric("final_train_loss", train_losses[-1])
    mlflow.log_metric("accuracy", test_accuracy)
    mlflow.log_metric("recall", test_recall)
    for epoch, (loss, acc) in enumerate(zip(train_losses, train_accs)):
        mlflow.log_metric("train_loss", loss, step=epoch)
        mlflow.log_metric("train_accuracy", acc, step=epoch)

## Export to ONNX and verify

Mirrors the CAI reference exactly: `torch.onnx.export(...)` on the raw trained model, opset 11, dynamic batch, input tensor `input`, output tensor `output`. The endpoint returns raw logits — apply softmax + argmax client-side (or in `nba_app.py` when swapping over from XGBoost).

Because the model was trained on `StandardScaler`-normalized features, the client is responsible for applying the same scaling before sending the payload. `scaler.mean_` and `scaler.scale_` are logged as MLflow params on the training run for that.

In [ ]:
import inspect


def convert_to_onnx(model, input_size, onnx_path):
    """Convert PyTorch model to ONNX format (mirrors the CAI reference)."""
    model.eval()
    dummy_input = torch.randn(input_size)
    # PyTorch >= 2.5 supports a ``dynamo`` kwarg; force the legacy
    # TorchScript exporter to keep the export path identical to the CAI
    # reference and avoid the onnxscript path.
    export_kwargs = dict(
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {0: "batch_size"},
        },
    )
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        export_kwargs["dynamo"] = False
    torch.onnx.export(model, dummy_input, onnx_path, **export_kwargs)
    print(f"Model exported to ONNX format: {onnx_path}")
    return onnx_path


def verify_onnx_model(onnx_path, test_data):
    """Verify ONNX model works correctly."""
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    ort_session = ort.InferenceSession(onnx_path)
    ort_inputs = {ort_session.get_inputs()[0].name: test_data[:5].numpy()}
    ort_outputs = ort_session.run(None, ort_inputs)
    print(f"ONNX model verification successful. Output shape: {ort_outputs[0].shape}")
    return True


onnx_path = "nba_risk_pytorch.onnx"
convert_to_onnx(model, input_size=(1, input_size), onnx_path=onnx_path)
verify_onnx_model(onnx_path, X_test_tensor)

## Register both artifacts to the CAI Model Registry

Same pattern as the reference: log the raw PyTorch model (`nba-risk-pytorch`) and the ONNX artifact (`nba-risk-onnx-pytorch`) in one MLflow run. The ONNX one is what CAI Inference serves; point the deploy notebook at it.

In [ ]:
with mlflow.start_run():
    # 1) Raw PyTorch model.
    print("Logging PyTorch model to MLflow...")
    mlflow.pytorch.log_model(
        model,
        "nba-risk-pytorch",
        registered_model_name=REGISTERED_MODEL_NAME_PYTORCH,
    )

    # 2) ONNX artifact — this is what CAI Inference actually serves.
    print("Logging ONNX model to MLflow...")
    onnx_model = onnx.load(onnx_path)
    mlflow.onnx.log_model(
        onnx_model,
        "nba-risk-onnx-pytorch",
        registered_model_name=REGISTERED_MODEL_NAME_ONNX,
    )

    print("Models logged successfully!")